# ContraBin quickstart

This notebook walks through the ContraBin pipeline end-to-end on synthetic data. It runs in < 30 s on a laptop CPU and requires no network access.

Steps:

1. Build synthetic triplet data.
2. Wire up tokenizer + dataloader.
3. Instantiate a tiny ContraBin model.
4. Pre-train for one full curriculum pass.
5. Extract binary embeddings and measure retrieval metrics.

In [ ]:
import tempfile
from pathlib import Path

from contrabin.config import (
    ContraBinConfig, CurriculumConfig, DataConfig,
    ModelConfig, OptimConfig, TrainingConfig,
)
from contrabin.data.datasets import TripletDataset, build_synthetic_triplets
from contrabin.data.loaders import TripletCollator, build_dataloader, build_tokenizer
from contrabin.training.trainer import PretrainTrainer
from contrabin.tasks.binary_retrieval import (
    extract_binary_embeddings, evaluate_retrieval,
)

## 1 & 2. Synthetic triplets + dataloader

In [ ]:
td = Path(tempfile.mkdtemp())
train_path, val_path = td / 'train.jsonl', td / 'val.jsonl'
build_synthetic_triplets(train_path, n=64, seed=0)
build_synthetic_triplets(val_path,   n=16, seed=1)

cfg = ContraBinConfig(
    data=DataConfig(train_path=train_path, val_path=val_path,
                    source_max_length=32, binary_max_length=32,
                    comment_max_length=12, num_workers=0),
    model=ModelConfig(encoder_name='contrabin-tiny',
                      hidden_dim=32, projection_dim=16, dropout=0.0, temperature=0.1),
    optim=OptimConfig(lr=1e-3, head_lr=1e-3, warmup_steps=0, scheduler='constant'),
    training=TrainingConfig(
        batch_size=8, eval_batch_size=8, device='cpu',
        output_dir=td / 'outputs', log_every_n_steps=100,
        curriculum=CurriculumConfig(primary_epochs=1, linear_epochs=1, nonlinear_epochs=1),
    ),
)
tokenizer = build_tokenizer(cfg.model.encoder_name, vocab_size=64)
collator = TripletCollator(
    tokenizer=tokenizer,
    source_max_length=cfg.data.source_max_length,
    binary_max_length=cfg.data.binary_max_length,
    comment_max_length=cfg.data.comment_max_length,
)
train_loader = build_dataloader(TripletDataset(train_path), collator, batch_size=cfg.training.batch_size)
val_loader   = build_dataloader(TripletDataset(val_path),   collator, batch_size=cfg.training.eval_batch_size, shuffle=False)
print('train size:', len(train_loader.dataset), 'val size:', len(val_loader.dataset))

## 3-4. Train

In [ ]:
trainer = PretrainTrainer(cfg)
state = trainer.fit(train_loader, val_loader)
state.history

## 5. Retrieval evaluation

The synthetic templates come from four program families (add / sub / mul / id), and `build_synthetic_triplets` tags each record with its family via `metadata['idx'] % 4`. We use that as a stand-in label so retrieval metrics are meaningful.

In [ ]:
class _Labeled:
    def __init__(self, inner):
        self.inner = inner
    def __len__(self):
        return len(self.inner)
    def __getitem__(self, i):
        r = dict(self.inner[i]); meta = dict(r.get('metadata', {}))
        meta['label'] = meta.get('idx', i) % 4
        r['metadata'] = meta; return r

retrieval_loader = build_dataloader(_Labeled(TripletDataset(val_path)), collator, batch_size=8, shuffle=False)
emb, labels = extract_binary_embeddings(trainer.model, retrieval_loader, device='cpu')
evaluate_retrieval(emb, labels)